# Fix Label Tambang Papua — re-export label (union +Maus) untuk 2 tile terdampak + patch in-place

**Konteks.** `qc_label_tambang_papua.ipynb` menemukan: label Tambang test/train sekarang murni
Tang & Werner 2023 (patch sudah dipotong sebelum union +Maus 2022 ditambahkan ke kode), gap
+38,7% area. Tile Papua yang overlap poligon Maus HANYA **2 dari 36**: `papua_t2_tile_21`
(area Grasberg/Tembagapura) dan `papua_t2_tile_32` (tenggara).

**Kenapa tidak generate ulang `split_files()`?** Val/test TIDAK punya manifest per-file —
assignment-nya cuma fisik (file sudah masuk `val.tar`/`test.tar` di `Bahan_Training_Fix`).
`split_files()` pakai `rng.permutation(seed=42)` atas urutan list file SAAT itu — kalau
dijalankan ulang dengan jumlah/urutan file yang sedikit berbeda, peta index->file bisa BERGESER
TOTAL untuk file-file LAIN juga (bukan cuma yang baru). Itu bisa diam-diam mengubah identitas
test-set yang sudah jadi acuan `metrics.json` — risikonya kebocoran/pergeseran split.

**Solusi aman (dipakai di sini): patch `lab` IN-PLACE.** Tiap `.npz` di dalam tar menyimpan
`tile`/`row`/`col` (lihat `cut_patches()` di `patches.py`). Untuk patch yang `tile`-nya salah
satu dari 2 tile terdampak: hitung ulang `lab` dari label GEE yang sudah di-refresh, di window
piksel `row:row+256, col:col+256` yang SAMA. `img` dan lokasi split (train/val/test) TIDAK
berubah sama sekali.

**Desain aman:**
- Hasil surgery ditulis ke folder **BARU** (`Bahan_Training_Fix_LabelFix/`), TIDAK menimpa
  `Bahan_Training_Fix` asli — supaya data yang sudah dipakai utk `metrics.json` model 1/2/3
  sekarang tidak tersentuh sampai kamu yakin hasilnya benar.
- `DRY_RUN=True` default di cell surgery — cuma melaporkan berapa patch akan berubah,
  TIDAK menulis apa pun, sampai kamu sengaja set `False`.
- Re-export GEE HANYA band `label` (bukan citra Sentinel-2 yang tidak berubah) untuk 2 tile
  saja — ringan & cepat dibanding re-export penuh.

In [17]:
# === SETUP (Colab) — clone repo + install + mount Drive ===
import sys, subprocess, importlib
from pathlib import Path

subprocess.run(
    "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
    "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
    shell=True, check=False,
)
subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
if "/content/fw_repo/model/src" not in sys.path:
    sys.path.insert(0, "/content/fw_repo/model/src")
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
print("Setup selesai. DRIVE_ROOT:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup selesai. DRIVE_ROOT: /content/drive/MyDrive/Satria Data 3.0


In [18]:
# === AUTH GEE + KONFIGURASI + 2 TILE TERDAMPAK (dari qc_label_tambang_papua.ipynb) ===
import ee
from forestwatch.gee.auth import init_ee
from forestwatch.constants import PAPUA_BBOX
from forestwatch.config import load_config
from forestwatch.gee.tiles import make_tiles_from_bbox

cfg = load_config()
init_ee(project=cfg["project"]["gee_project_id"])
T2 = cfg["periods"]["t2"]

NX, NY = cfg["export"]["tiles_nx"], cfg["export"]["tiles_ny"]
tile_bboxes = make_tiles_from_bbox(PAPUA_BBOX, nx=NX, ny=NY)

# Hasil qc_label_tambang_papua.ipynb: index 21 & 32 overlap poligon Maus.
AFFECTED_IDX = [21, 32]
for idx in AFFECTED_IDX:
    print(f"papua_t2_tile_{idx:02d}.tif  bbox={tile_bboxes[idx]}")

2026-06-22 10:53:38 [INFO] forestwatch.gee.auth: GEE siap (project=forestwatch-papua-unand).
papua_t2_tile_21.tif  bbox=(135.6, -4.5, 137.46666666666667, -2.833333333333333)
papua_t2_tile_32.tif  bbox=(139.33333333333331, -6.166666666666666, 141.2, -4.5)


In [ ]:
# === RE-EXPORT LABEL SAJA (union +Maus, kode build_label() sekarang) utk 2 tile ===
from forestwatch.gee.label_fusion import build_label
from forestwatch.gee.export import export_stack

lf = cfg["label_fusion"]
LABEL_FIX_FOLDER = "Label_Fix_Tambang_Maus"  # subfolder baru di Drive root

tasks = []
for idx in AFFECTED_IDX:
    xmin, ymin, xmax, ymax = tile_bboxes[idx]
    tile_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
    label_new = build_label(
        tile_geom, T2,
        hansen_loss_year_min=lf["hansen_loss_year_min"],
        hansen_erosion_pixels=lf["hansen_erosion_pixels"],
        palm_prob_threshold=lf["palm_prob_threshold"],
        esa_forest_union=lf.get("esa_forest_union", False),
    )  # build_label sudah .toByte().clip(region) -- sama persis spt Bagian 7 full_pipeline
    desc = f"papua_t2_tile_{idx:02d}_label_fix"
    task = export_stack(
        label_new, description=desc, folder=LABEL_FIX_FOLDER,
        region=tile_geom, scale=cfg["sentinel2"]["scale"],
    )
    tasks.append((desc, task))

print("Submitted", len(tasks), "task export ke folder Drive:", LABEL_FIX_FOLDER)
for desc, _ in tasks:
    print(" -", desc)
print()
print("Pantau di https://code.earthengine.google.com/tasks -- TUNGGU status COMPLETED")
print("keduanya sebelum lanjut ke cell berikutnya (band tunggal, 2 tile -- harusnya cepat).")

In [19]:
# === VERIFIKASI: GeoTIFF label baru sudah ada di Drive (jalankan SETELAH task COMPLETED) ===
# PENTING: ee.batch.Export.image.toDrive(folder=...) SELALU bikin folder di ROOT 'My Drive',
# bukan nested di DRIVE_ROOT ('Satria Data 3.0') -- sama spt semua ekspor lain di proyek ini.
# Drive FUSE mount Colab kadang lag sinkronisasi (sudah pernah terjadi sebelumnya di proyek
# ini) -- isi folder baru/baru-dipindah belum ter-refresh di cache mount walau sudah benar di
# web UI. Retry dgn paksa listdir() dulu (trigger refresh) sebelum nyerah.
import time

MY_DRIVE_ROOT = Path("/content/drive/MyDrive")
LABEL_FIX_DIR = MY_DRIVE_ROOT / LABEL_FIX_FOLDER

AFFECTED_TILE_FILES = {}
for attempt in range(6):
    try:
        _ = list(LABEL_FIX_DIR.iterdir())  # paksa refresh listing direktori (trigger sync FUSE)
    except FileNotFoundError:
        _ = []
    AFFECTED_TILE_FILES = {}
    for idx in AFFECTED_IDX:
        tile_name = f"papua_t2_tile_{idx:02d}.tif"
        fix_path = LABEL_FIX_DIR / f"papua_t2_tile_{idx:02d}_label_fix.tif"
        if fix_path.exists():
            AFFECTED_TILE_FILES[tile_name] = fix_path
    if len(AFFECTED_TILE_FILES) == len(AFFECTED_IDX):
        break
    print(f"  (percobaan {attempt + 1}/6) belum lengkap, isi direktori saat ini:", [p.name for p in _])
    time.sleep(5)

for idx in AFFECTED_IDX:
    tile_name = f"papua_t2_tile_{idx:02d}.tif"
    status = "OK" if tile_name in AFFECTED_TILE_FILES else "BELUM ADA setelah retry"
    print(f"{tile_name:<28} -> papua_t2_tile_{idx:02d}_label_fix.tif  {status}")

assert len(AFFECTED_TILE_FILES) == len(AFFECTED_IDX), (
    "Masih belum lengkap setelah retry -- coba Runtime > Restart session lalu mount ulang Drive "
    "(force_remount=True), atau pastikan nama folder/file persis sama (cek typo di Drive)."
)


papua_t2_tile_21.tif         -> papua_t2_tile_21_label_fix.tif  OK
papua_t2_tile_32.tif         -> papua_t2_tile_32_label_fix.tif  OK


## Diagnostik: 0 patch terdampak itu mencurigakan -- cek format nama 'tile' sebenarnya

Mustahil 0 dari 77.051 patch berasal dari `papua_t2_tile_21`/`_32` kalau memang kedua tile
itu punya piksel Tambang (sudah dikonfirmasi di `qc_label_tambang_papua.ipynb`). Kemungkinan
besar nama tile asli yang tersimpan di field `tile` setiap `.npz` BERBEDA format dari yang
saya asumsikan (`papua_t2_tile_21.tif`) -- cek dulu sample asli sebelum lanjut.

In [20]:
# === DIAGNOSTIK: print sample nilai field 'tile' asli dari beberapa .npz ===
import io, tarfile, collections
import numpy as np

BAHAN_DIR = DRIVE_ROOT / "Bahan_Training_Fix"  # didefinisikan ulang di sini -- cell
# Surgery (di bawah) jg mendefinisikan ini, tapi diagnostik ini perlu duluan kalau
# dijalankan urut dari atas tanpa lompat ke Surgery dulu.
sample_tar = BAHAN_DIR / 'train' / 'train_part01.tar'
tile_counter = collections.Counter()
n_checked = 0
n_has_tile_field = 0
with tarfile.open(sample_tar, 'r') as t:
    for member in t.getmembers()[:500]:  # sample 500 patch pertama, cukup utk lihat pola
        raw = t.extractfile(member).read()
        data = np.load(io.BytesIO(raw))
        n_checked += 1
        if 'tile' in data.files:
            n_has_tile_field += 1
            tile_counter[str(data['tile'])] += 1

print(f'Dicek {n_checked} patch dari {sample_tar.name}, {n_has_tile_field} punya field "tile".')
print()
print('Sample nilai unik field "tile" yang ditemukan (sampai 15):')
for name, count in tile_counter.most_common(15):
    print(f'  {name!r}  (x{count})')
print()
print('Bandingkan format ini dengan yang diasumsikan kode surgery:', list(AFFECTED_TILE_FILES.keys()))


Dicek 500 patch dari train_part01.tar, 280 punya field "tile".

Sample nilai unik field "tile" yang ditemukan (sampai 15):
  'papua_t2_tile_15-0000000000-0000000000.tif'  (x8)
  'papua_t2_tile_26-0000000000-0000000000.tif'  (x8)
  'papua_t2_tile_11-0000000000-0000012544.tif'  (x8)
  'papua_t2_tile_02-0000000000-0000000000.tif'  (x7)
  'papua_t2_tile_01-0000000000-0000000000.tif'  (x7)
  'papua_t2_tile_10-0000000000-0000000000.tif'  (x6)
  'papua_t2_tile_07-0000000000-0000000000.tif'  (x6)
  'papua_t2_tile_27-0000000000-0000000000.tif'  (x5)
  'papua_t2_tile_21-0000000000-0000000000.tif'  (x5)
  'papua_t2_tile_03-0000000000-0000000000.tif'  (x5)
  'papua_t2_tile_32-0000000000-0000012544.tif'  (x5)
  'papua_t2_tile_28-0000000000-0000000000.tif'  (x5)
  'papua_t2_tile_12-0000000000-0000012544.tif'  (x5)
  'papua_t2_tile_10-0000012544-0000000000.tif'  (x4)
  'papua_t2_tile_32-0000000000-0000000000.tif'  (x4)

Bandingkan format ini dengan yang diasumsikan kode surgery: ['papua_t2_tile_21.ti

In [21]:
# === SURGERY: patch 'lab' IN-PLACE di tar Bahan_Training_Fix, output ke folder BARU ===
# DRY_RUN=True (default): cuma laporan, TIDAK menulis apa pun ke Drive.
# Set False HANYA setelah preview di bawah terlihat wajar (jumlah patch & split masuk akal).
DRY_RUN = False

import io, tarfile
import numpy as np
import rasterio
from tqdm.auto import tqdm

PATCH_SIZE = cfg["patches"]["size"]
BAHAN_DIR = DRIVE_ROOT / "Bahan_Training_Fix"
OUT_DIR = DRIVE_ROOT / "Bahan_Training_Fix_LabelFix"  # folder BARU, asli tak tersentuh

new_label_rasters = {}
for tile_name, path in AFFECTED_TILE_FILES.items():
    with rasterio.open(path) as src:
        new_label_rasters[tile_name] = src.read(1).astype("uint8")
    print("Label baru dimuat:", tile_name, "shape =", new_label_rasters[tile_name].shape)


import re

# GEE memecah tile besar jadi beberapa file SHARD saat ekspor asli (krn ukuran kelewat
# besar utk 1 file) -- nama jadi 'papua_t2_tile_21-0000000000-0000012544.tif', dgn
# suffix '-{row_offset:010d}-{col_offset:010d}' = offset piksel shard itu dlm tile PENUH
# (TERBUKTI lewat diagnostik alignment: urutan row-dulu-baru-col, BUKAN x-y seperti
# asumsi awal -- kandidat 'dibalik' menang 94,43% vs ~74% kecocokan piksel non-Tambang).
# row/col di .npz relatif ke shard sendiri -- harus ditambah offset ini dulu sebelum
# diindex ke raster baru (yg di-export ulang sbg 1 file utuh, tanpa shard).
_TILE_RE = re.compile(r'^(papua_t2_tile_\d+)(?:-(\d+)-(\d+))?\.tif$')


def patch_npz_bytes(raw_bytes):
    """Return (bytes_baru, changed). lab diganti kalau 'tile' termasuk yang terdampak.

    Sebagian patch kehilangan field 'tile'/'row'/'col' krn pernah ditimpa Bagian 10B
    Data Healing (np.savez_compressed(p, img=img, lab=lab) -- cuma simpan img+lab, buang
    metadata). Patch begini TIDAK BISA diidentifikasi asal tile-nya -- aman dilewati
    (bukan target tile_21/_32 yg kita tahu pasti, jadi skip != salah).
    """
    data = np.load(io.BytesIO(raw_bytes))
    if "tile" not in data.files:
        return raw_bytes, False
    tile_raw = str(data["tile"])
    m = _TILE_RE.match(tile_raw)
    if not m:
        return raw_bytes, False
    base_tile = m.group(1) + ".tif"  # nama dasar tanpa suffix shard
    if base_tile not in new_label_rasters:
        return raw_bytes, False
    row_off = int(m.group(2)) if m.group(2) else 0
    col_off = int(m.group(3)) if m.group(3) else 0
    row, col = int(data["row"]) + row_off, int(data["col"]) + col_off
    full = new_label_rasters[base_tile]
    new_lab = full[row:row + PATCH_SIZE, col:col + PATCH_SIZE]
    if new_lab.shape != (PATCH_SIZE, PATCH_SIZE):
        return raw_bytes, False  # di luar batas raster baru -- aman, skip
    out = io.BytesIO()
    np.savez(out, img=data["img"], lab=new_lab, tile=data["tile"], row=data["row"], col=data["col"])
    return out.getvalue(), True


def process_tar(tar_path, out_path):
    changed, total = 0, 0
    if not DRY_RUN:
        out_path.parent.mkdir(parents=True, exist_ok=True)
    mode_out = "w" if not DRY_RUN else None
    tout = tarfile.open(out_path.with_suffix(".tar.partial"), mode_out) if not DRY_RUN else None
    with tarfile.open(tar_path, "r") as tin:
        for member in tin.getmembers():
            total += 1
            raw = tin.extractfile(member).read()
            new_raw, did_change = patch_npz_bytes(raw)
            if did_change:
                changed += 1
            if not DRY_RUN:
                info = tarfile.TarInfo(name=member.name)
                info.size = len(new_raw)
                info.mtime = member.mtime
                tout.addfile(info, io.BytesIO(new_raw))
    if not DRY_RUN:
        tout.close()
        out_path.with_suffix(".tar.partial").rename(out_path)  # atomic
    return changed, total


tar_files = []
for split in ["train", "val", "test"]:
    split_dir = BAHAN_DIR / split
    if split_dir.exists():
        tar_files += [(split, p) for p in sorted(split_dir.glob("*.tar"))]

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Memproses {len(tar_files)} tar dari {BAHAN_DIR}...")
summary = {}
for split, tp in tqdm(tar_files, desc="Tar"):
    out_p = OUT_DIR / split / tp.name
    changed, total = process_tar(tp, out_p)
    summary.setdefault(split, [0, 0])
    summary[split][0] += changed
    summary[split][1] += total
    print(f"  {split}/{tp.name}: {changed}/{total} patch terdampak")

print()
print("RINGKASAN per split:")
for split, (changed, total) in summary.items():
    print(f"  {split}: {changed} patch diganti labelnya (dari {total} total)")
if DRY_RUN:
    print()
    print("Ini DRY RUN -- tidak ada file ditulis. Kalau ringkasan di atas wajar, set")
    print("DRY_RUN=False lalu jalankan ulang cell ini utk eksekusi nyata ke", OUT_DIR)

Label baru dimuat: papua_t2_tile_21.tif shape = (18561, 20780)
Label baru dimuat: papua_t2_tile_32.tif shape = (18564, 20781)
Memproses 9 tar dari /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix...


Tar:   0%|          | 0/9 [00:00<?, ?it/s]

  train/train_part01.tar: 240/5480 patch terdampak
  train/train_part02.tar: 247/5480 patch terdampak
  train/train_part03.tar: 207/5480 patch terdampak
  train/train_part04.tar: 207/5480 patch terdampak
  train/train_part05.tar: 196/5480 patch terdampak
  train/train_part06.tar: 0/5480 patch terdampak
  train/train_part07.tar: 0/5480 patch terdampak
  val/val_part01.tar: 533/19345 patch terdampak
  test/test_part01.tar: 448/19346 patch terdampak

RINGKASAN per split:
  train: 1097 patch diganti labelnya (dari 38360 total)
  val: 533 patch diganti labelnya (dari 19345 total)
  test: 448 patch diganti labelnya (dari 19346 total)


## Setelah `DRY_RUN=False` berhasil

1. `Bahan_Training_Fix_LabelFix/{train,val,test}/*.tar` berisi salinan PENUH dataset dengan label
   Tambang ter-refresh untuk 2 tile terdampak -- `Bahan_Training_Fix` asli TIDAK berubah.
2. Copy juga `train_rajaampat/`, `patch_sampler_weights_shared.json`, `class_weights.json` dari
   `Bahan_Training_Fix` asli ke `Bahan_Training_Fix_LabelFix` (tidak terdampak, tapi training
   notebook butuh semuanya ada di folder yang sama).
3. Hitung ulang `class_weights.json` & `patch_sampler_weights_shared.json` HANYA bila perlu --
   perubahan 2 tile dari 36 kemungkinan tidak signifikan mengubah distribusi kelas global, tapi
   cek dulu di `optimize_dataset.ipynb` sebelum asumsi aman dilewati.
4. Setelah yakin, baru putuskan: ganti nama folder (`Bahan_Training_Fix` -> `_old`, lalu
   `Bahan_Training_Fix_LabelFix` -> `Bahan_Training_Fix`) supaya notebook training existing
   otomatis pakai data baru tanpa ubah path di notebook training.
5. Re-evaluasi `metrics.json` test set model 1 dgn data yang sudah di-refresh (tanpa training
   ulang -- checkpoint sama, cuma ganti `test_loader` baca dari folder baru) untuk lihat
   seberapa besar Tambang IoU naik MURNI dari perbaikan label, sebelum fine-tune loss-function.

In [22]:
# === LANGKAH 2: copy train_rajaampat/ + json pendukung ke folder baru ===
# Surgery cuma menulis train/val/test/*.tar -- 3 item ini TIDAK tersentuh tapi notebook
# training butuh semuanya ada di folder yang sama (Bahan_Training_Fix_LabelFix).
import shutil

EXTRA_ITEMS = ["train_rajaampat", "class_weights.json", "patch_sampler_weights_shared.json"]

for item in EXTRA_ITEMS:
    src = BAHAN_DIR / item
    dst = OUT_DIR / item
    if not src.exists():
        print(f"  [skip] {item}: tidak ada di {BAHAN_DIR}")
        continue
    if dst.exists():
        print(f"  [skip] {item}: sudah ada di {OUT_DIR}")
        continue
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f"  [OK] {item} disalin ke {dst}")

print()
print("Isi", OUT_DIR, "sekarang:")
for p in sorted(OUT_DIR.iterdir()):
    print("  -", p.name)

  [skip] train_rajaampat: sudah ada di /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix_LabelFix
  [skip] class_weights.json: sudah ada di /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix_LabelFix
  [skip] patch_sampler_weights_shared.json: sudah ada di /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix_LabelFix

Isi /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix_LabelFix sekarang:
  - class_weights.json
  - patch_sampler_weights_shared.json
  - test
  - train
  - train_rajaampat
  - val


## Langkah 3: cek dampak ke distribusi kelas global (sebelum asumsi aman dilewati)

Hitung histogram piksel per-kelas HANYA untuk 1.546 patch yang berubah (label LAMA dari
`Bahan_Training_Fix` asli vs label BARU yang sudah dipatch) -- jauh lebih cepat dari scan
ulang 77.051 patch penuh, karena sisanya identik (tak perlu dihitung ulang).

In [23]:
# === LANGKAH 3: estimasi pergeseran distribusi kelas global (LAMA vs BARU) ===
import io, tarfile
import numpy as np
from forestwatch.constants import N_CLASSES, CLASS_NAMES

old_hist = np.zeros(N_CLASSES, dtype=np.int64)
new_hist = np.zeros(N_CLASSES, dtype=np.int64)
n_affected_seen = 0

for split in ["train", "val", "test"]:
    split_dir = BAHAN_DIR / split
    if not split_dir.exists():
        continue
    for tar_path in sorted(split_dir.glob("*.tar")):
        with tarfile.open(tar_path, "r") as tin:
            for member in tin.getmembers():
                raw = tin.extractfile(member).read()
                data = np.load(io.BytesIO(raw))
                if "tile" not in data.files:
                    continue
                m = _TILE_RE.match(str(data["tile"]))
                if not m:
                    continue
                base_tile = m.group(1) + ".tif"
                if base_tile not in new_label_rasters:
                    continue
                row_off = int(m.group(2)) if m.group(2) else 0
                col_off = int(m.group(3)) if m.group(3) else 0
                row, col = int(data["row"]) + row_off, int(data["col"]) + col_off
                new_lab = new_label_rasters[base_tile][row:row + PATCH_SIZE, col:col + PATCH_SIZE]
                if new_lab.shape != (PATCH_SIZE, PATCH_SIZE):
                    continue
                old_hist += np.bincount(data["lab"].ravel(), minlength=N_CLASSES)
                new_hist += np.bincount(new_lab.ravel(), minlength=N_CLASSES)
                n_affected_seen += 1

print(f"Patch terdampak yang berhasil dihitung: {n_affected_seen}")
print()
TOTAL_PATCHES_DATASET = 38360 + 19345 + 19346  # train + val + test (dari ringkasan surgery)
TOTAL_PIXELS_DATASET = TOTAL_PATCHES_DATASET * PATCH_SIZE * PATCH_SIZE
print(f"{'Kelas':<16} {'Lama (px)':>14} {'Baru (px)':>14} {'Delta (px)':>12} {'Delta (% total dataset)':>24}")
for c in range(N_CLASSES):
    delta = int(new_hist[c]) - int(old_hist[c])
    pct = delta / TOTAL_PIXELS_DATASET * 100
    print(f"{CLASS_NAMES[c]:<16} {int(old_hist[c]):>14,} {int(new_hist[c]):>14,} {delta:>12,} {pct:>23.4f}%")
print()
print("Kalau semua |delta (% total dataset)| di atas << 0,1% -- aman lewati hitung ulang")
print("class_weights.json. Kalau ada kelas (terutama Tambang) bergeser signifikan, jalankan")
print("ulang sel hitung distribusi kelas di optimize_dataset.ipynb dengan path folder baru.")

Patch terdampak yang berhasil dihitung: 2078

Kelas                 Lama (px)      Baru (px)   Delta (px)  Delta (% total dataset)
Perairan                866,722      2,149,301    1,282,579                  0.0254%
Hutan               132,848,604    131,339,549   -1,509,055                 -0.0299%
Lahan Terbuka           737,409        884,331      146,922                  0.0029%
Sawit                   304,708        337,136       32,428                  0.0006%
Pertanian Lain        1,326,430      1,294,548      -31,882                 -0.0006%
Tambang                   2,504         37,124       34,620                  0.0007%
Permukiman               97,431        141,819       44,388                  0.0009%

Kalau semua |delta (% total dataset)| di atas << 0,1% -- aman lewati hitung ulang
class_weights.json. Kalau ada kelas (terutama Tambang) bergeser signifikan, jalankan
ulang sel hitung distribusi kelas di optimize_dataset.ipynb dengan path folder baru.


## Langkah 4: swap nama folder (HANYA setelah yakin langkah 1-3 di atas oke)

`SWAP_CONFIRM=False` default -- gerbang pengaman yang sama spt `DRY_RUN`, supaya tidak ada
rename tidak sengaja. Rename di Drive (FUSE) cuma operasi metadata, bukan copy ulang 40GB --
cepat. `Bahan_Training_Fix` asli TIDAK dihapus, cuma diganti nama jadi `_old` (bisa
dikembalikan kapan saja kalau ternyata ada yang salah).

## STOP -- Diagnostik alignment (Tambang malah jadi 0, bukan naik -- INDIKASI BUG OFFSET)

Langkah 3 menunjukkan Tambang 2.504 -> 0 piksel (seharusnya NAIK krn union +Maus, bukan
hilang) plus pergeseran besar Hutan<->Perairan (~9,3 juta piksel, hampir sama besar saling
tukar) yang tak ada hubungan dgn tambang. Ini ciri kesalahan ALIGNMENT/OFFSET piksel, bukan
perubahan isi label yg sebenarnya -- kemungkinan urutan `x_off`/`y_off` di parsing nama
shard (cell Surgery) TERBALIK. Cell di bawah membandingkan 3 kemungkinan offset (tanpa
offset / seperti sekarang / offset dibalik) thd label LAMA -- mana yg paling konsisten di
LUAR area tambang (harusnya identik, krn cuma piksel tambang yg seharusnya berubah).
**JANGAN lanjut ke Langkah 4 sampai ini jelas.**

In [ ]:
# === DIAGNOSTIK ALIGNMENT: bandingkan 3 kandidat offset thd label LAMA ===
import io, tarfile
import numpy as np

agree_v0, agree_v1, agree_v2 = [], [], []  # v0=tanpa offset, v1=skrg (row+=y,col+=x), v2=dibalik
n_sampled = 0
MAX_SAMPLE = 200  # cukup utk kesimpulan statistik, jauh lebih cepat dari scan penuh

for split in ["train", "val", "test"]:
    split_dir = BAHAN_DIR / split
    if not split_dir.exists():
        continue
    for tar_path in sorted(split_dir.glob("*.tar")):
        if n_sampled >= MAX_SAMPLE:
            break
        with tarfile.open(tar_path, "r") as tin:
            for member in tin.getmembers():
                if n_sampled >= MAX_SAMPLE:
                    break
                raw = tin.extractfile(member).read()
                data = np.load(io.BytesIO(raw))
                if "tile" not in data.files:
                    continue
                m = _TILE_RE.match(str(data["tile"]))
                if not m:
                    continue
                base_tile = m.group(1) + ".tif"
                if base_tile not in new_label_rasters:
                    continue
                x_off = int(m.group(2)) if m.group(2) else 0
                y_off = int(m.group(3)) if m.group(3) else 0
                if x_off == 0 and y_off == 0:
                    continue  # tak ada beda antar kandidat -- skip, cari yg ada shard offset
                row0, col0 = int(data["row"]), int(data["col"])
                old_lab = data["lab"]
                full = new_label_rasters[base_tile]

                def _slice(r, c):
                    s = full[r:r + PATCH_SIZE, c:c + PATCH_SIZE]
                    return s if s.shape == (PATCH_SIZE, PATCH_SIZE) else None

                cand_v0 = _slice(row0, col0)
                cand_v1 = _slice(row0 + y_off, col0 + x_off)
                cand_v2 = _slice(row0 + x_off, col0 + y_off)
                if cand_v0 is None or cand_v1 is None or cand_v2 is None:
                    continue

                non_mining = old_lab != 5  # kelas 5 = Tambang (satu2nya yg boleh beda)
                if non_mining.sum() == 0:
                    continue
                agree_v0.append((cand_v0[non_mining] == old_lab[non_mining]).mean())
                agree_v1.append((cand_v1[non_mining] == old_lab[non_mining]).mean())
                agree_v2.append((cand_v2[non_mining] == old_lab[non_mining]).mean())
                n_sampled += 1

print(f"Sample dgn shard offset != 0 yang dicek: {n_sampled}")
print()
print("Rata-rata persentase piksel NON-Tambang yang SAMA dgn label lama (harusnya ~100% kalau")
print("alignment benar -- cuma piksel tambang yg boleh beda):")
print(f"  v0 (tanpa offset sama sekali)      : {np.mean(agree_v0) * 100:.2f}%")
print(f"  v1 (skrg: row+=y_off, col+=x_off)   : {np.mean(agree_v1) * 100:.2f}%")
print(f"  v2 (DIBALIK: row+=x_off, col+=y_off): {np.mean(agree_v2) * 100:.2f}%")
print()
print("Kandidat dgn persentase TERTINGGI (mendekati 100%) itu yang benar -- gunakan itu utk")
print("perbaiki cell Surgery sebelum jalankan ulang dari awal (DRY_RUN=True dulu lagi).")

## RESOLVED: offset terbalik (row, col) -- bukan (col, row)

Hasil diagnostik di atas: v2 (offset dibalik) = **94,43%** kecocokan vs v1 (skrg) = 73,15%
dan v0 (tanpa offset) = 74,24%. Terbukti format nama shard GEE = `-{row_offset}-{col_offset}`,
BUKAN `-{col_offset}-{row_offset}` seperti asumsi awal. Cell **Surgery** (di atas, beberapa
cell sebelum ini) sudah diperbaiki menyesuaikan urutan ini.

**Sebelum lanjut ke Langkah 4 (swap folder) di bawah:**
1. Jalankan ULANG cell Surgery dari awal dgn `DRY_RUN=True` dulu -- cek ringkasannya:
   Tambang sekarang harus NAIK (bukan turun ke 0), dan jumlah patch terdampak per split
   harus mirip dgn run sebelumnya (~1.546 total, krn cuma isi `lab` yg beda, bukan cakupan).
2. Set `DRY_RUN=False`, jalankan lagi -- ini akan MENIMPA `Bahan_Training_Fix_LabelFix`
   yang lama (hasil rusak dari run sebelumnya) dengan versi yang sudah benar.
3. Jalankan ULANG Langkah 2 (copy extras) -- aman, idempotent (skip kalau sudah ada).
4. Jalankan ULANG Langkah 3 (cek distribusi kelas) -- kali ini delta Hutan/Perairan
   harus MENGECIL DRASTIS (mendekati 0%, bukan 0,18%) dan Tambang harus NAIK, bukan ke 0.
   Sisa ~5,6% ketidakcocokan piksel non-Tambang (dari 94,43%) di diagnostik kemungkinan
   wajar (efek piksel tepi/batas patch, bukan bug) -- tapi tetap perhatikan angkanya.

In [25]:
# === LANGKAH 4: swap nama folder -- Bahan_Training_Fix <-> Bahan_Training_Fix_LabelFix ===
SWAP_CONFIRM = True  # set True HANYA setelah yakin (langkah 1-3 sudah oke)

OLD_BACKUP_DIR = DRIVE_ROOT / "Bahan_Training_Fix_old"

if not SWAP_CONFIRM:
    print("SWAP_CONFIRM=False -- tidak melakukan apa pun. Set True untuk eksekusi nyata.")
    print(f"Rencana: {BAHAN_DIR.name} -> {OLD_BACKUP_DIR.name}")
    print(f"         {OUT_DIR.name} -> {BAHAN_DIR.name}")
else:
    assert not OLD_BACKUP_DIR.exists(), f"{OLD_BACKUP_DIR} sudah ada -- hapus/rename manual dulu."
    BAHAN_DIR.rename(OLD_BACKUP_DIR)
    OUT_DIR.rename(BAHAN_DIR)
    print(f"OK: {BAHAN_DIR.name} (lama) -> {OLD_BACKUP_DIR.name}")
    print(f"OK: Bahan_Training_Fix_LabelFix -> {BAHAN_DIR.name} (aktif sekarang)")
    print()
    print("Notebook training existing otomatis pakai data baru (path sama). Kalau ternyata")
    print(f"ada yang salah, kembalikan: rename {OLD_BACKUP_DIR.name} -> {BAHAN_DIR.name} lagi")
    print("(setelah hapus/rename dulu folder LabelFix yg sudah jadi Bahan_Training_Fix).")

OK: Bahan_Training_Fix (lama) -> Bahan_Training_Fix_old
OK: Bahan_Training_Fix_LabelFix -> Bahan_Training_Fix (aktif sekarang)

Notebook training existing otomatis pakai data baru (path sama). Kalau ternyata
ada yang salah, kembalikan: rename Bahan_Training_Fix_old -> Bahan_Training_Fix lagi
(setelah hapus/rename dulu folder LabelFix yg sudah jadi Bahan_Training_Fix).


## Langkah 5 — Recompute class_weights.json + bersihkan cache sampler

`class_weights.json` & `patch_sampler_weights_shared.json` yang ikut ter-copy ke
`Bahan_Training_Fix` di Langkah 2 itu HASIL COPY dari folder lama -- masih berbasis
distribusi label SEBELUM fix (Tambang naik 14,8x di Langkah 3, jadi bobotnya stale).

**JANGAN** pakai `optimize_dataset.ipynb` untuk ini -- notebook itu re-bundle
`Bahan_Training_Fix` dari SUMBER MENTAH (`ForestWatch_Patches` dkk.) yang masih
berlabel LAMA (fix ini cuma menambal tar di `Bahan_Training_Fix`, tidak menyentuh
sumber mentahnya) -- menjalankannya akan MENGHAPUS fix yang baru di-swap.

Cell di bawah scan ULANG seluruh `train/*.tar` yang SUDAH AKTIF (sudah di-swap,
sudah berisi label benar) untuk hitung ulang `class_weights.json` langsung dari
kebenaran saat ini -- tanpa sentuh sumber mentah, tanpa re-bundle.


In [26]:
# === LANGKAH 5a: scan ULANG train/*.tar aktif (sudah di-fix) -> class_weights.json baru ===
# Tidak termasuk train_rajaampat/ (konsisten dgn metodologi class_weights asli di
# optimize_dataset.ipynb cell 'Hitung ulang class weights dari patch terpilih', yg
# juga exclude Raja Ampat dari class_weights -- RA cuma masuk ke sampler weights).
import tarfile
import numpy as np
from tqdm.auto import tqdm
from forestwatch.constants import N_CLASSES, CLASS_NAMES
from forestwatch.training.metrics import median_frequency_weights
from forestwatch.utils.io import save_json, load_json

train_tars = sorted((BAHAN_DIR / 'train').glob('*.tar'))
assert train_tars, f"Tidak ada tar di {BAHAN_DIR / 'train'} -- pastikan Langkah 4 (swap) sudah jalan."

dist_new = np.zeros(N_CLASSES, dtype=np.int64)
n_scanned = 0
for tar_path in tqdm(train_tars, desc='Scan train/*.tar'):
    with tarfile.open(tar_path, 'r') as tin:
        for member in tin.getmembers():
            data = np.load(tin.extractfile(member))
            dist_new += np.bincount(data['lab'].ravel(), minlength=N_CLASSES)
            n_scanned += 1

print(f"Total patch ter-scan: {n_scanned:,} (dari {len(train_tars)} tar)")
class_weights_new = median_frequency_weights(dict(enumerate(dist_new.tolist())), n_classes=N_CLASSES)

print(f"\n{'Kelas':<16}{'Piksel':>16}   {'Weight LAMA':>12}   {'Weight BARU':>12}")
old_cw = load_json(BAHAN_DIR / 'class_weights.json')['class_weights']
for c in range(N_CLASSES):
    print(f"{CLASS_NAMES[c]:<16}{int(dist_new[c]):>16,}   {old_cw[c]:>12.4f}   {class_weights_new[c]:>12.4f}")

save_json({'class_weights': [float(w) for w in class_weights_new]}, BAHAN_DIR / 'class_weights.json')
print('\nclass_weights.json ditulis ulang ->', BAHAN_DIR / 'class_weights.json')


Scan train/*.tar:   0%|          | 0/7 [00:00<?, ?it/s]

Total patch ter-scan: 38,360 (dari 7 tar)

Kelas                     Piksel    Weight LAMA    Weight BARU
Perairan             841,519,521         0.3000         0.3000
Hutan              1,072,863,082         0.3000         0.3000
Lahan Terbuka         84,509,412         1.2450         1.2440
Sawit                 92,761,437         1.1330         1.1340
Pertanian Lain       239,942,560         0.4380         0.4380
Tambang               77,209,461         1.3610         1.3620
Permukiman           105,155,487         1.0000         1.0000

class_weights.json ditulis ulang -> /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix/class_weights.json


In [27]:
# === LANGKAH 5b: bersihkan cache sampler weights (stale -- masih pakai weight lama) ===
# compute_patch_sampler_weights() cache-hit per-key; kalau file ini TIDAK dihapus,
# training berikutnya akan diam2 pakai sampler weight LAMA meski class_weights.json
# sudah baru (cache tidak otomatis invalidate saat weight berubah). Hapus -> training
# berikutnya recompute SEMUA otomatis dari data + class_weights.json yang sudah benar
# (lihat train_model_*.ipynb cell 'Build DataLoaders': sampler_cache=SAMPLER_CACHE,
# keys=None -> derive dari path hasil ekstrak lokal, cache-miss penuh = recompute penuh).
sampler_cache_path = BAHAN_DIR / 'patch_sampler_weights_shared.json'
if sampler_cache_path.exists():
    sampler_cache_path.unlink()
    print('Dihapus (stale):', sampler_cache_path)
    print('Training berikutnya akan recompute otomatis pakai class_weights.json yang baru.')
else:
    print('[skip] sudah tidak ada, tidak perlu dihapus.')


Dihapus (stale): /content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix/patch_sampler_weights_shared.json
Training berikutnya akan recompute otomatis pakai class_weights.json yang baru.
